# March Madness 2026 Predictor

This notebook trains separate **Random Forest** classifiers for the Men's and Women's NCAA Tournament to predict the probability that Team A wins any given matchup. The final output is a `submission.csv` compatible with the Kaggle *March Machine Learning Mania 2026* competition.

**High-level flow:**
1. Load raw game data and compute per-team ELO ratings across all historical seasons
2. Build a labeled dataset of past tournament games, enriching each matchup with regular-season stats, ELO, seeding, Massey rankings, and prior tournament performance
3. Train using a **walk-forward** (expanding-window) strategy and evaluate accuracy + log loss season by season
4. Generate win-probability predictions for every possible 2026 tournament matchup pair
5. Fill in neutral (50/50) probabilities for non-tournament teams and write `submission.csv`

---

## Imports

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import os

import matplotlib.pyplot as plt
import pandas as pd  # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    log_loss,
)
from tqdm import tqdm

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/march-machine-learning-mania-2026/Conferences.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/WNCAATourneyDetailedResults.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/WRegularSeasonCompactResults.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/MNCAATourneySeedRoundSlots.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/MRegularSeasonDetailedResults.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/MNCAATourneyCompactResults.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/MGameCities.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/WSecondaryTourneyCompactResults.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/WGameCities.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/MSeasons.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/WNCAATourneySlots.csv
/kaggle/input/competitions/march-machine-learning

## Configuration

Defines the global constants used throughout the notebook:
- `start_data_season` — the first season included in training data (Men's from 2003, Women's from 2009)
- `classifiers` — pre-configured `RandomForestClassifier` instances with hyperparameters tuned separately for each gender
- `columns` — the full ordered list of feature + metadata column names shared by all DataFrames
- `season` — the target prediction season (2026)

In [2]:
start_data_season = {"M": 2003, "W": 2009}
classifiers = {
    "M": RandomForestClassifier(
        max_depth=80,
        max_features="sqrt",
        min_samples_leaf=4,
        n_estimators=1600,
        random_state=0,
    ),
    "W": RandomForestClassifier(
        max_depth=100,
        max_features=None,
        min_samples_leaf=4,
        min_samples_split=5,
        n_estimators=2000,
        random_state=0,
    ),
}
columns = [
    # Game Metadata
    "Season",
    "TeamA",
    "TeamB",
    "TeamNameA",
    "TeamNameB",
    # Team A Data
    "ASeedNum",
    "ALastTournPct",
    "AELO",
    "AMasseyRank",
    "ARegSznPct",
    # Team A Previous Game Stats
    "APts",
    "AFG",
    "AFG3",
    "AFT",
    "AAst",
    "ATO",
    "AOR",
    "ADR",
    "AStl",
    "ABlk",
    # Team A Previous Game Stats (Opponent)
    "APtsOpp",
    "AOppFG",
    "AOppFG3",
    "AOppFT",
    "AOppAst",
    "AOppTO",
    "AOppOR",
    "AOppDR",
    "AOppStl",
    "AOppBlk",
    # Team B Data
    "BSeedNum",
    "BLastTournPct",
    "BELO",
    "BMasseyRank",
    "BRegSznPct",
    # Team B Previous Game Stats
    "BPts",
    "BFG",
    "BFG3",
    "BFT",
    "BAst",
    "BTO",
    "BOR",
    "BDR",
    "BStl",
    "BBlk",
    # Team B Previous Game Stats (Opponent)
    "BPtsOpp",
    "BOppFG",
    "BOppFG3",
    "BOppFT",
    "BOppAst",
    "BOppTO",
    "BOppOR",
    "BOppDR",
    "BOppStl",
    "BOppBlk",
    # Match Winner
    "Winner",
]
season = 2026

## Team Information

Loads the `{gender}Teams.csv` file for both Men's and Women's competitions. The resulting DataFrames (`m_teams_info_df`, `w_teams_info_df`) are used throughout the notebook to resolve numeric team IDs to human-readable team names.

In [3]:
def get_teams_info(gender):
    teams_file = f"/kaggle/input/competitions/march-machine-learning-mania-2026/{gender}Teams.csv"
    teams_df = pd.read_csv(teams_file)
    display(teams_df)
    return teams_df


m_teams_info_df = get_teams_info("M")
w_teams_info_df = get_teams_info("W")

,TeamID,TeamName,FirstD1Season,LastD1Season
0,1101,Abilene Chr,2014,2026
1,1102,Air Force,1985,2026
2,1103,Akron,1985,2026
3,1104,Alabama,1985,2026
4,1105,Alabama A&M,2000,2026
...,...,...,...,...
376,1477,East Texas A&M,2023,2026
377,1478,Le Moyne,2024,2026
378,1479,Mercyhurst,2025,2026
379,1480,West Georgia,2025,2026


,TeamID,TeamName
0,3101,Abilene Chr
1,3102,Air Force
2,3103,Akron
3,3104,Alabama
4,3105,Alabama A&M
...,...,...
374,3477,East Texas A&M
375,3478,Le Moyne
376,3479,Mercyhurst
377,3480,West Georgia


## ELO Rating System

Implements a margin-of-victory–adjusted ELO system:
- `get_k` — K-factor that scales with the winning margin and is dampened when the stronger team wins big, preventing runaway ratings
- `get_e_team` — standard ELO expected-score formula
- `update_elo` — applies the ELO delta to both teams in-place inside a shared dict
- `initialize_teams_elo_dict` — seeds every team to a starting ELO of 1500

All teams start at 1500 and their ratings evolve game-by-game through the entire historical record before each season's tournament.

In [4]:
def get_k(vic_margin, elo_diff_winner):
    return 20 * ((vic_margin + 3) ** 0.8) / (7.5 + 0.006 * elo_diff_winner)


def get_e_team(team_elo, opp_team_elo):
    return 1 / (1 + 10 ** ((opp_team_elo - team_elo) / 400))


def update_elo(
    elo_dict, winning_team_id, losing_team_id, winning_team_pts, losing_team_pts
):
    winning_team_elo = elo_dict[winning_team_id]
    losing_team_elo = elo_dict[losing_team_id]

    vic_margin = winning_team_pts - losing_team_pts
    elo_diff_winner = winning_team_elo - losing_team_elo
    elo_dict[winning_team_id] = round(
        get_k(vic_margin, elo_diff_winner)
        * (1 - get_e_team(winning_team_elo, losing_team_elo))
        + winning_team_elo,
        2,
    )
    elo_dict[losing_team_id] = round(
        get_k(vic_margin, elo_diff_winner)
        * (0 - get_e_team(losing_team_elo, winning_team_elo))
        + losing_team_elo,
        2,
    )


def initialize_teams_elo_dict(teams_df, full_games_df):
    full_games_df["WTeamELO"] = 1500
    full_games_df["LTeamELO"] = 1500

    elo_dict = dict()
    for team_id in teams_df["TeamID"]:
        elo_dict[team_id] = 1500

    return elo_dict

## Data Loading & ELO Calculation

`get_files_by_competition` loads all necessary CSVs for a given gender and computes ELO ratings across every historical game:

1. Reads regular-season, tournament, and seed CSV files; generates percentage-based shooting columns (`FG%`, `FG3%`, `FT%`)
2. Merges everything into a single chronologically sorted `full_games_df`
3. Iterates through every game in order — snapshotting each team's ELO *before* the game is played (the feature reflects form entering the game), then updating ELO after
4. Batch-assigns ELO snapshots back to the source DataFrames via `df.loc` (significantly faster than per-row `df.at` writes)

Returns: `teams_df`, `reg_szn_df` (with ELO), `tourn_df` (with ELO), `seed_df`, and the final `teams_elo` dict.

In [5]:
def get_seed_number(seed):
    if "a" in seed or "b" in seed:
        return 17
    return int(seed[1:])


def generate_numeric_cols(df, team):
    df[f"{team}FG"] = (df[f"{team}FGM"] * 100) / df[f"{team}FGA"]
    df[f"{team}FG3"] = (df[f"{team}FGM3"] * 100) / df[f"{team}FGA3"]
    df[f"{team}FT"] = (df[f"{team}FTM"] * 100) / df[f"{team}FTA"]


def generate_game_id(game):
    return f"{game['Season']}_{game['DayNum']}_{game['WTeamID']}_{game['LTeamID']}"


def get_files_by_competition(gender):
    # Getting teams df
    teams_file = f"/kaggle/input/competitions/march-machine-learning-mania-2026/{gender}TeamConferences.csv"
    teams_df = pd.read_csv(teams_file)

    # Getting regular season games df
    reg_szn_file = f"/kaggle/input/competitions/march-machine-learning-mania-2026/{gender}RegularSeasonDetailedResults.csv"
    reg_szn_df = pd.read_csv(reg_szn_file)
    reg_szn_df["GameId"] = reg_szn_df.apply(generate_game_id, axis=1)
    reg_szn_df["IsTourn"] = 0
    reg_szn_df.set_index("GameId", inplace=True)
    generate_numeric_cols(reg_szn_df, "W")
    generate_numeric_cols(reg_szn_df, "L")

    # Getting tournament season games df
    tourn_file = f"/kaggle/input/competitions/march-machine-learning-mania-2026/{gender}NCAATourneyCompactResults.csv"
    tourn_df = pd.read_csv(tourn_file)
    tourn_df["GameId"] = tourn_df.apply(generate_game_id, axis=1)
    tourn_df["IsTourn"] = 1
    tourn_df.set_index("GameId", inplace=True)

    # Getting seeding df
    seed_file = f"/kaggle/input/competitions/march-machine-learning-mania-2026/{gender}NCAATourneySeeds.csv"
    seed_df = pd.read_csv(seed_file)
    seed_df["SeedNum"] = seed_df.Seed.apply(lambda x: get_seed_number(x))

    game_defining_cols = [
        "Season",
        "DayNum",
        "WTeamID",
        "WScore",
        "LTeamID",
        "LScore",
        "WLoc",
        "NumOT",
        "IsTourn",
    ]
    full_games_df = pd.concat(
        [df[game_defining_cols] for df in [reg_szn_df, tourn_df]]
    ).sort_values(by=["Season", "DayNum"])

    teams_elo = initialize_teams_elo_dict(teams_df, full_games_df)

    # Accumulate ELO snapshots per game index, then batch-assign (avoids slow per-row df.at writes)
    reg_elo_w, reg_elo_l = {}, {}
    tourn_elo_w, tourn_elo_l = {}, {}

    for row in tqdm(full_games_df.itertuples(), total=len(full_games_df)):
        if row.IsTourn:
            tourn_elo_w[row.Index] = teams_elo[row.WTeamID]
            tourn_elo_l[row.Index] = teams_elo[row.LTeamID]
        else:
            reg_elo_w[row.Index] = teams_elo[row.WTeamID]
            reg_elo_l[row.Index] = teams_elo[row.LTeamID]

        update_elo(teams_elo, row.WTeamID, row.LTeamID, row.WScore, row.LScore)

    # Batch assign — much faster than df.at inside the loop
    if reg_elo_w:
        reg_szn_df.loc[list(reg_elo_w.keys()), "WTeamELO"] = list(reg_elo_w.values())
        reg_szn_df.loc[list(reg_elo_l.keys()), "LTeamELO"] = list(reg_elo_l.values())
    if tourn_elo_w:
        tourn_df.loc[list(tourn_elo_w.keys()), "WTeamELO"] = list(tourn_elo_w.values())
        tourn_df.loc[list(tourn_elo_l.keys()), "LTeamELO"] = list(tourn_elo_l.values())

    display(full_games_df)

    return teams_df, reg_szn_df, tourn_df, seed_df, teams_elo

## Regular Season Statistics Cache

**`get_reg_szn_stats`** (reference implementation) computes a team's season averages by filtering the full regular-season DataFrame for each call.

**`build_reg_szn_stats_cache`** vectorizes this computation across all `(season, team)` pairs at once using `groupby`, producing a `dict` keyed by `(season, team_id)`. This cache is built once per pipeline run and queried in O(1) per game, eliminating thousands of redundant DataFrame filter operations during dataset construction and prediction.

The 21 values returned per team are:
`RegSznWin%`, `Pts`, `FG%`, `FG3%`, `FT%`, `Ast`, `TO`, `OR`, `DR`, `Stl`, `Blk` (team) + the same 10 opponent-allowed averages.

In [6]:
def get_reg_szn_stats(season, team, reg_szn_df):
    w_games = (
        reg_szn_df[(reg_szn_df["Season"] == season) & (reg_szn_df["WTeamID"] == team)]
        .rename(
            columns={
                "WFG": "FG",
                "WFG3": "FG3",
                "WFT": "FT",
                "WAst": "Ast",
                "WTO": "TO",
                "WOR": "OR",
                "WDR": "DR",
                "WStl": "Stl",
                "WBlk": "Blk",
                "WPF": "PF",
                "WScore": "Score",
                "WTeamELO": "ELO",
                "LFG": "OppFG",
                "LFG3": "OppFG3",
                "LFT": "OppFT",
                "LAst": "OppAst",
                "LTO": "OppTO",
                "LOR": "OppOR",
                "LDR": "OppDR",
                "LStl": "OppStl",
                "LBlk": "OppBlk",
                "LPF": "OppPF",
                "LScore": "OppScore",
                "LTeamELO": "OppELO",
            }
        )
        .reset_index(drop=True)
    )
    w_games["Won"] = 1

    l_games = (
        reg_szn_df[(reg_szn_df["Season"] == season) & (reg_szn_df["LTeamID"] == team)]
        .rename(
            columns={
                "LFG": "FG",
                "LFG3": "FG3",
                "LFT": "FT",
                "LAst": "Ast",
                "LTO": "TO",
                "LOR": "OR",
                "LDR": "DR",
                "LStl": "Stl",
                "LBlk": "Blk",
                "LPF": "PF",
                "LScore": "Score",
                "LTeamELO": "ELO",
                "WFG": "OppFG",
                "WFG3": "OppFG3",
                "WFT": "OppFT",
                "WAst": "OppAst",
                "WTO": "OppTO",
                "WOR": "OppOR",
                "WDR": "OppDR",
                "WStl": "OppStl",
                "WBlk": "OppBlk",
                "WPF": "OppPF",
                "WScore": "OppScore",
                "WTeamELO": "OppELO",
            }
        )
        .reset_index(drop=True)
    )
    l_games["Won"] = 0

    games = pd.concat([w_games, l_games], axis=0, ignore_index=True)

    reg_szn_pct = (len(w_games) * 100) / len(games)

    return [
        reg_szn_pct,
        # Team stats
        games["Score"].mean(),
        games["FG"].mean(),
        games["FG3"].mean(),
        games["FT"].mean(),
        games["Ast"].mean(),
        games["TO"].mean(),
        games["OR"].mean(),
        games["DR"].mean(),
        games["Stl"].mean(),
        games["Blk"].mean(),
        # Opponent stats
        games["OppScore"].mean(),
        games["OppFG"].mean(),
        games["OppFG3"].mean(),
        games["OppFT"].mean(),
        games["OppAst"].mean(),
        games["OppTO"].mean(),
        games["OppOR"].mean(),
        games["OppDR"].mean(),
        games["OppStl"].mean(),
        games["OppBlk"].mean(),
    ]


def build_reg_szn_stats_cache(reg_szn_df):
    """Vectorized precomputation of regular-season stats for all (season, team) pairs.
    Returns a dict mapping (season, team_id) -> list of 21 stat values (same order as get_reg_szn_stats).
    """
    stat_cols_team = ["Score", "FG", "FG3", "FT", "Ast", "TO", "OR", "DR", "Stl", "Blk"]
    stat_cols_opp = [
        "OppScore",
        "OppFG",
        "OppFG3",
        "OppFT",
        "OppAst",
        "OppTO",
        "OppOR",
        "OppDR",
        "OppStl",
        "OppBlk",
    ]

    w_df = reg_szn_df.rename(
        columns={
            "WFG": "FG",
            "WFG3": "FG3",
            "WFT": "FT",
            "WAst": "Ast",
            "WTO": "TO",
            "WOR": "OR",
            "WDR": "DR",
            "WStl": "Stl",
            "WBlk": "Blk",
            "WScore": "Score",
            "LFG": "OppFG",
            "LFG3": "OppFG3",
            "LFT": "OppFT",
            "LAst": "OppAst",
            "LTO": "OppTO",
            "LOR": "OppOR",
            "LDR": "OppDR",
            "LStl": "OppStl",
            "LBlk": "OppBlk",
            "LScore": "OppScore",
        }
    )[["Season", "WTeamID"] + stat_cols_team + stat_cols_opp].rename(
        columns={"WTeamID": "TeamID"}
    )

    l_df = reg_szn_df.rename(
        columns={
            "LFG": "FG",
            "LFG3": "FG3",
            "LFT": "FT",
            "LAst": "Ast",
            "LTO": "TO",
            "LOR": "OR",
            "LDR": "DR",
            "LStl": "Stl",
            "LBlk": "Blk",
            "LScore": "Score",
            "WFG": "OppFG",
            "WFG3": "OppFG3",
            "WFT": "OppFT",
            "WAst": "OppAst",
            "WTO": "OppTO",
            "WOR": "OppOR",
            "WDR": "OppDR",
            "WStl": "OppStl",
            "WBlk": "OppBlk",
            "WScore": "OppScore",
        }
    )[["Season", "LTeamID"] + stat_cols_team + stat_cols_opp].rename(
        columns={"LTeamID": "TeamID"}
    )

    wins_count = w_df.groupby(["Season", "TeamID"]).size()
    all_games = pd.concat([w_df, l_df], axis=0, ignore_index=True)
    total_count = all_games.groupby(["Season", "TeamID"]).size()
    reg_szn_pct = (wins_count / total_count * 100).fillna(0).to_dict()

    means = all_games.groupby(["Season", "TeamID"])[
        stat_cols_team + stat_cols_opp
    ].mean()

    cache = {}
    for (s, t), row in means.iterrows():
        cache[(s, t)] = [
            reg_szn_pct.get((s, t), 0),
            row["Score"],
            row["FG"],
            row["FG3"],
            row["FT"],
            row["Ast"],
            row["TO"],
            row["OR"],
            row["DR"],
            row["Stl"],
            row["Blk"],
            row["OppScore"],
            row["OppFG"],
            row["OppFG3"],
            row["OppFT"],
            row["OppAst"],
            row["OppTO"],
            row["OppOR"],
            row["OppDR"],
            row["OppStl"],
            row["OppBlk"],
        ]
    return cache

## Massey Rankings, Tournament History & Seed Lookup

Three additional precomputed O(1) lookup caches:

**`build_massey_lookup`** — averages the final pre-tournament ordinal rank from the POM, MAS, and RPI systems. Falls back to `MASSEY_DEFAULT_RANK = 175` (approximate median D1 rank) for teams with no data. Only available for Men's.

**`build_last_tourn_pct_cache`** — each team's win percentage in the *previous* year's tournament, keyed by `(current_season, team_id)`. Teams that didn't make the prior tournament default to 0.

**`build_seed_lookup`** — maps `(season, team_id)` → numeric seed; play-in teams are mapped to seed 17.

In [7]:
MASSEY_SYSTEMS = ["POM", "MAS", "RPI"]
MASSEY_DEFAULT_RANK = 175  # neutral fallback (midpoint of ~350 D1 teams)


def build_massey_lookup(massey_df):
    """Precompute (season, team_id) -> mean ordinal rank across POM/MAS/RPI."""
    if massey_df is None:
        return None

    df = massey_df[massey_df["SystemName"].isin(MASSEY_SYSTEMS)]

    # For each season/team/system, keep only the latest available day's rank
    latest = df.loc[
        df.groupby(["Season", "TeamID", "SystemName"])["RankingDayNum"].idxmax()
    ]
    lookup = latest.groupby(["Season", "TeamID"])["OrdinalRank"].mean()

    return lookup.to_dict()


def get_massey_rank(season, team, massey_lookup):
    if massey_lookup is None:
        return MASSEY_DEFAULT_RANK

    return massey_lookup.get((season, team), MASSEY_DEFAULT_RANK)


def build_last_tourn_pct_cache(tourn_df):
    """Precompute last-tournament win % keyed by (next_season, team_id)."""
    w_counts = (
        tourn_df.groupby(["Season", "WTeamID"])
        .size()
        .rename("Wins")
        .reset_index()
        .rename(columns={"WTeamID": "TeamID"})
    )
    l_counts = (
        tourn_df.groupby(["Season", "LTeamID"])
        .size()
        .rename("Losses")
        .reset_index()
        .rename(columns={"LTeamID": "TeamID"})
    )
    merged = pd.merge(w_counts, l_counts, on=["Season", "TeamID"], how="outer").fillna(
        0
    )
    merged["Pct"] = merged["Wins"] * 100 / (merged["Wins"] + merged["Losses"])
    # Key by (next_season, team_id) so callers can use the current season directly
    return {
        (int(row.Season) + 1, int(row.TeamID)): row.Pct
        for row in merged.itertuples(index=False)
    }


def get_last_tourn_pct(season, team, last_tourn_pct_cache):
    return last_tourn_pct_cache.get((season, team), 0)


def build_seed_lookup(seed_df):
    """Precompute (season, team_id) -> seed number."""
    return seed_df.set_index(["Season", "TeamID"])["SeedNum"].to_dict()

## Feature Assembly & Dataset Construction

**`get_game_stats`** assembles the full feature vector for a single matchup by hitting all precomputed caches (ELO, regular-season stats, seed, Massey rank, last tournament percentage). Teams are always ordered so the lower team ID is Team A, ensuring a consistent representation.

**`build_dataset`** iterates over every historical NCAA Tournament game across all training seasons, assigns winner labels (`'A'` or `'B'`), and accumulates all rows into a labeled DataFrame using the schema defined in `columns`.

In [8]:
def set_team_name(team_id, teams_df):
    return teams_df[teams_df["TeamID"] == team_id].reset_index().loc[0, "TeamName"]


def get_game_stats(
    season,
    team_a,
    team_b,
    team_a_elo,
    team_b_elo,
    reg_szn_stats_cache,
    last_tourn_pct_cache,
    seed_lookup,
    teams_info_df,
    massey_lookup,
):
    a_reg_szn_stats = reg_szn_stats_cache.get((season, team_a), [0] * 21)
    b_reg_szn_stats = reg_szn_stats_cache.get((season, team_b), [0] * 21)

    a_seed = seed_lookup.get((season, team_a), 24)
    b_seed = seed_lookup.get((season, team_b), 24)

    a_last_tourn_pct = get_last_tourn_pct(season, team_a, last_tourn_pct_cache)
    b_last_tourn_pct = get_last_tourn_pct(season, team_b, last_tourn_pct_cache)

    a_massey_rank = get_massey_rank(season, team_a, massey_lookup)
    b_massey_rank = get_massey_rank(season, team_b, massey_lookup)

    stats_a = [a_seed, a_last_tourn_pct, team_a_elo, a_massey_rank] + a_reg_szn_stats
    stats_b = [b_seed, b_last_tourn_pct, team_b_elo, b_massey_rank] + b_reg_szn_stats

    team_a_name = set_team_name(team_a, teams_info_df)
    team_b_name = set_team_name(team_b, teams_info_df)

    return [season, team_a, team_b, team_a_name, team_b_name] + stats_a + stats_b


def build_dataset(
    start_data_season,
    tourn_df,
    end_season,
    teams_info_df,
    massey_lookup,
    reg_szn_stats_cache,
    last_tourn_pct_cache,
    seed_lookup,
):
    data = []

    for season in tqdm(
        range(start_data_season + 1, end_season + 1), desc="Processing seasons"
    ):
        tourney_games = tourn_df[tourn_df["Season"] == season].reset_index(drop=True)

        for _, g in tourney_games.iterrows():
            team_a = min([g["WTeamID"], g["LTeamID"]])
            team_b = max([g["WTeamID"], g["LTeamID"]])

            if team_a == g["WTeamID"]:
                winner = "A"
                team_a_elo = g["WTeamELO"]
                team_b_elo = g["LTeamELO"]
            else:
                winner = "B"
                team_a_elo = g["LTeamELO"]
                team_b_elo = g["WTeamELO"]

            game_stats = get_game_stats(
                season,
                team_a,
                team_b,
                team_a_elo,
                team_b_elo,
                reg_szn_stats_cache,
                last_tourn_pct_cache,
                seed_lookup,
                teams_info_df,
                massey_lookup,
            )
            data.append(game_stats + [winner])

    data_df = pd.DataFrame(data, columns=columns)
    display(data_df)

    return data_df

## Model Training & Walk-Forward Evaluation

**`train_model`** fits the gender-specific `RandomForestClassifier` on the provided training set.

**`predict_seasons`** evaluates the model using an **expanding-window walk-forward** strategy:
- For each season from `start_data_season + 2` to the current season, trains on *all prior seasons* and tests on the *current season*
- Reports per-season accuracy and log loss, then overall means
- Optionally displays a confusion matrix per season when `detailed_results=True`
- Returns the classifier trained on the **full historical dataset**, which is then used to generate 2026 predictions

In [9]:
def train_model(x_train, y_train, gender):
    classifier = classifiers[gender]
    classifier.fit(x_train, y_train)
    return classifier


def predict_seasons(gender, start_data_season, data_df, detailed_results):
    acc_sum = 0
    ll_sum = 0
    seasons_count = 0

    for season_itr in range(start_data_season + 2, season + 1):
        data_train = data_df[(data_df["Season"] < season_itr)].reset_index(drop=True)
        data_test = data_df[(data_df["Season"] == season_itr)].reset_index(drop=True)

        if not len(data_test):
            continue

        x_train = data_train.drop(["TeamNameA", "TeamNameB", "Winner"], axis=1)
        y_train = data_train.loc[:, "Winner"]

        classifier = train_model(x_train, y_train, gender)

        x_test = data_test.drop(["TeamNameA", "TeamNameB", "Winner"], axis=1)
        y_test = data_test.loc[:, "Winner"]

        predictions = classifier.predict(x_test)
        probs = classifier.predict_proba(x_test)

        acc = accuracy_score(y_test, predictions)
        ll = log_loss(y_test, probs)
        acc_sum += acc
        ll_sum += ll
        seasons_count += 1

        if detailed_results:
            print(f"\nResults for {gender} season {season_itr}:")
            print("Accuracy:", acc)
            print("Log Loss:", ll)

            cm = confusion_matrix(y_test, predictions)
            print("Confusion matrix:")
            cm_disp = ConfusionMatrixDisplay(
                confusion_matrix=cm, display_labels=classifier.classes_
            )
            cm_disp.plot()
            plt.show()

    print(f"\n\n{gender} Mean Accuracy: {acc_sum/seasons_count}")
    print(f"{gender} Mean Log Loss:  {ll_sum/seasons_count}")

    return classifier

## Current Season Predictions

`get_current_predictions` generates win probabilities for every possible head-to-head pairing among the 2026 tournament teams (all seeded teams in `seed_df` for that season):
- Uses the classifier trained on all historical data
- Looks up each team's pre-computed ELO, regular-season stats, seed, and rankings
- Returns a DataFrame with `AProb` (Team A win probability), `BProb`, and the hard `Pred` label per matchup

In [10]:
def get_current_predictions(
    classifier,
    seed_df,
    team_elo_dict,
    teams_info_df,
    massey_lookup,
    reg_szn_stats_cache,
    last_tourn_pct_cache,
    seed_lookup,
):
    data = []
    teams = seed_df[seed_df["Season"] == season].TeamID.unique()
    teams.sort()

    for idx, team_a in tqdm(enumerate(teams), total=len(teams)):
        for idx_b in range(idx + 1, len(teams)):
            team_b = teams[idx_b]

            team_a_elo = team_elo_dict[team_a]
            team_b_elo = team_elo_dict[team_b]

            game_stats = get_game_stats(
                season,
                team_a,
                team_b,
                team_a_elo,
                team_b_elo,
                reg_szn_stats_cache,
                last_tourn_pct_cache,
                seed_lookup,
                teams_info_df,
                massey_lookup,
            )
            data.append(game_stats)

    data_df = pd.DataFrame(data, columns=columns[:-1])
    data_to_predict = data_df.drop(["TeamNameA", "TeamNameB"], axis=1)

    probs = classifier.predict_proba(data_to_predict)
    pred = classifier.predict(data_to_predict)

    data_df["AProb"] = probs[:, 0]
    data_df["BProb"] = probs[:, 1]
    data_df["Pred"] = pred

    return data_df

## Pipeline

`pipeline` is the top-level orchestrator. Given a gender, it:
1. Loads all data files and computes ELO ratings (`get_files_by_competition`)
2. Optionally builds the Massey ranking lookup (Men's only; Women's gets `None`)
3. Precomputes all stat, seed, and tournament-history caches in one vectorized pass
4. Builds the historical labeled dataset (`build_dataset`)
5. Runs walk-forward training and evaluation, printing accuracy and log loss (`predict_seasons`)
6. Generates 2026 matchup predictions (`get_current_predictions`)

Returns `(current_szn_df, teams_df, team_elo_dict)`.

In [11]:
def pipeline(gender, detailed_results=False):
    teams_info_df = m_teams_info_df if gender == "M" else w_teams_info_df
    teams_df, reg_szn_df, tourn_df, seed_df, team_elo_dict = get_files_by_competition(
        gender
    )

    if gender == "M":
        raw_massey_df = pd.read_csv(
            "/kaggle/input/competitions/march-machine-learning-mania-2026/MMasseyOrdinals.csv"
        )
        massey_lookup = build_massey_lookup(raw_massey_df)
    else:
        massey_lookup = None  # No Massey data available for women's

    # Build lookup caches once — avoids repeated per-game DataFrame filtering
    reg_szn_stats_cache = build_reg_szn_stats_cache(reg_szn_df)
    last_tourn_pct_cache = build_last_tourn_pct_cache(tourn_df)
    seed_lookup = build_seed_lookup(seed_df)

    data_df = build_dataset(
        start_data_season[gender],
        tourn_df,
        season,
        teams_info_df,
        massey_lookup,
        reg_szn_stats_cache,
        last_tourn_pct_cache,
        seed_lookup,
    )
    classifier = predict_seasons(
        gender, start_data_season[gender], data_df, detailed_results
    )
    current_szn_df = get_current_predictions(
        classifier,
        seed_df,
        team_elo_dict,
        teams_info_df,
        massey_lookup,
        reg_szn_stats_cache,
        last_tourn_pct_cache,
        seed_lookup,
    )
    display(current_szn_df)
    return current_szn_df, teams_df, team_elo_dict

## Run — Men's Tournament

Executes the full pipeline for the **Men's** competition. This is the longer-running of the two due to the larger training dataset (data from 2003 onward) and the higher `n_estimators` in the classifier. Results are stored in `m_current_szn_df`, `m_teams_df`, and `m_team_elo_dict`.

In [12]:
m_current_szn_df, m_teams_df, m_team_elo_dict = pipeline('M')

100%|██████████| 127114/127114 [00:00<00:00, 168154.94it/s]


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,IsTourn,WTeamELO,LTeamELO
GameId,,,,,,,,,,,
1985_136_1116_1234,1985,136,1116,63,1234,54,N,0,1,1500,1500
1985_136_1120_1345,1985,136,1120,59,1345,58,N,0,1,1500,1500
1985_136_1207_1250,1985,136,1207,68,1250,43,N,0,1,1500,1500
1985_136_1229_1425,1985,136,1229,58,1425,55,N,0,1,1500,1500
1985_136_1242_1325,1985,136,1242,49,1325,38,N,0,1,1500,1500
...,...,...,...,...,...,...,...,...,...,...,...
2026_132_1335_1463,2026,132,1335,88,1463,84,N,1,0,1500,1500
2026_132_1345_1276,2026,132,1345,80,1276,72,N,0,0,1500,1500
2026_132_1378_1455,2026,132,1378,70,1455,55,N,0,0,1500,1500


Processing seasons: 100%|██████████| 23/23 [00:02<00:00, 11.43it/s]


,Season,TeamA,TeamB,TeamNameA,TeamNameB,ASeedNum,ALastTournPct,AELO,AMasseyRank,ARegSznPct,...,BOppFG,BOppFG3,BOppFT,BOppAst,BOppTO,BOppOR,BOppDR,BOppStl,BOppBlk,Winner
0,2004,1197,1250,Florida A&M,Lehigh,17,0.000000,1456.14,271.000000,46.666667,...,41.168914,35.066209,65.753858,13.392857,13.892857,11.107143,23.321429,7.857143,3.428571,A
1,2004,1104,1356,Alabama,S Illinois,8,0.000000,1673.56,33.333333,58.620690,...,42.607419,30.758378,67.868298,12.448276,17.000000,11.241379,23.344828,6.103448,2.310345,A
2,2004,1163,1436,Connecticut,Vermont,2,66.666667,1827.12,7.333333,81.818182,...,42.943724,33.513813,65.487117,13.700000,13.000000,9.733333,23.833333,6.266667,4.133333,A
3,2004,1173,1177,Dayton,DePaul,10,0.000000,1669.42,50.000000,75.000000,...,42.353929,36.053952,69.107092,14.266667,12.400000,11.266667,19.666667,6.300000,2.700000,B
4,2004,1106,1181,Alabama St,Duke,16,0.000000,1454.01,258.333333,53.333333,...,40.967309,32.942131,71.332259,12.000000,17.718750,14.000000,20.125000,6.812500,3.562500,B
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1380,2025,1120,1277,Auburn,Michigan St,1,0.000000,2084.83,3.000000,84.848485,...,40.475051,27.783509,72.080299,12.484848,10.818182,7.727273,20.303030,6.000000,3.121212,A
1381,2025,1222,1397,Houston,Tennessee,1,66.666667,2158.88,3.333333,88.235294,...,38.143919,28.005336,73.238654,10.117647,10.676471,8.294118,19.205882,5.794118,2.941176,A
1382,2025,1120,1196,Auburn,Florida,1,0.000000,2091.01,3.000000,84.848485,...,39.794407,29.454945,71.585966,11.205882,11.441176,9.588235,20.941176,6.264706,3.117647,B
1383,2025,1181,1222,Duke,Houston,1,75.000000,2131.81,2.333333,91.176471,...,38.470130,30.963481,70.238897,10.794118,12.735294,7.352941,19.264706,4.941176,2.529412,B




M Mean Accuracy: 0.700264656829489
M Mean Log Loss:  0.5712166503423315


100%|██████████| 68/68 [00:03<00:00, 21.60it/s]


,Season,TeamA,TeamB,TeamNameA,TeamNameB,ASeedNum,ALastTournPct,AELO,AMasseyRank,ARegSznPct,...,BOppFT,BOppAst,BOppTO,BOppOR,BOppDR,BOppStl,BOppBlk,AProb,BProb,Pred
0,2026,1103,1104,Akron,Alabama,12,0.0,1745.26,55.333333,84.375000,...,68.713244,13.937500,9.281250,11.687500,24.531250,6.406250,3.687500,0.325654,0.674346,B
1,2026,1103,1112,Akron,Arizona,12,0.0,1745.26,55.333333,84.375000,...,71.393584,12.000000,11.823529,9.147059,18.852941,6.029412,3.705882,0.228426,0.771574,B
2,2026,1103,1116,Akron,Arkansas,12,0.0,1745.26,55.333333,84.375000,...,72.428785,14.441176,10.852941,9.794118,21.852941,5.029412,2.852941,0.324109,0.675891,B
3,2026,1103,1140,Akron,BYU,12,0.0,1745.26,55.333333,84.375000,...,68.706877,14.735294,11.088235,8.647059,21.794118,6.294118,2.970588,0.372570,0.627430,B
4,2026,1103,1155,Akron,Clemson,12,0.0,1745.26,55.333333,84.375000,...,72.945088,10.588235,10.764706,7.500000,22.147059,5.558824,3.294118,0.394364,0.605636,B
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2273,2026,1458,1465,Wisconsin,Cal Baptist,5,50.0,1963.70,24.000000,70.588235,...,72.736806,10.848485,11.333333,8.272727,21.181818,6.818182,3.878788,0.713308,0.286692,A
2274,2026,1458,1474,Wisconsin,Queens NC,5,50.0,1963.70,24.000000,70.588235,...,73.428350,13.727273,10.272727,9.545455,21.757576,6.454545,2.939394,0.671539,0.328461,A
2275,2026,1460,1465,Wright St,Cal Baptist,14,0.0,1508.56,129.333333,65.625000,...,72.736806,10.848485,11.333333,8.272727,21.181818,6.818182,3.878788,0.391674,0.608326,B
2276,2026,1460,1474,Wright St,Queens NC,14,0.0,1508.56,129.333333,65.625000,...,73.428350,13.727273,10.272727,9.545455,21.757576,6.454545,2.939394,0.486120,0.513880,B


## Run — Women's Tournament

Executes the full pipeline for the **Women's** competition (training data from 2009 onward; no Massey rankings available). Results are stored in `w_current_szn_df`, `w_teams_df`, and `w_team_elo_dict`.

In [13]:
w_current_szn_df, w_teams_df, w_team_elo_dict = pipeline('W')

100%|██████████| 88904/88904 [00:00<00:00, 167866.72it/s]


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,IsTourn,WTeamELO,LTeamELO
GameId,,,,,,,,,,,
1998_137_3104_3422,1998,137,3104,94,3422,46,H,0,1,1500,1500
1998_137_3112_3365,1998,137,3112,75,3365,63,H,0,1,1500,1500
1998_137_3163_3193,1998,137,3163,93,3193,52,H,0,1,1500,1500
1998_137_3198_3266,1998,137,3198,59,3266,45,H,0,1,1500,1500
1998_137_3203_3208,1998,137,3203,74,3208,72,A,0,1,1500,1500
...,...,...,...,...,...,...,...,...,...,...,...
2026_131_3471_3218,2026,131,3471,60,3218,48,N,0,0,1500,1500
2026_132_3158_3220,2026,132,3158,68,3220,56,N,0,0,1500,1500
2026_132_3192_3254,2026,132,3192,79,3254,57,H,0,0,1500,1500


Processing seasons: 100%|██████████| 17/17 [00:01<00:00, 11.80it/s]


,Season,TeamA,TeamB,TeamNameA,TeamNameB,ASeedNum,ALastTournPct,AELO,AMasseyRank,ARegSznPct,...,BOppFG,BOppFG3,BOppFT,BOppAst,BOppTO,BOppOR,BOppDR,BOppStl,BOppBlk,Winner
0,2010,3124,3201,Baylor,Fresno St,4,66.666667,1713.06,175,71.875000,...,40.451206,28.779373,64.660117,11.848485,19.878788,12.242424,24.818182,6.757576,2.606061,A
1,2010,3173,3395,Dayton,TCU,8,0.000000,1656.58,175,80.769231,...,36.537056,28.211876,67.463285,13.466667,20.666667,14.866667,25.333333,8.466667,3.500000,A
2,2010,3181,3214,Duke,Hampton,2,50.000000,1822.84,175,84.375000,...,38.237485,31.559370,64.247423,8.733333,22.133333,12.633333,26.133333,6.633333,2.266667,A
3,2010,3199,3256,Florida St,Louisiana Tech,3,50.000000,1739.62,175,83.333333,...,37.619004,28.069753,65.107530,10.741935,17.451613,13.709677,24.451613,8.193548,4.225806,A
4,2010,3207,3265,Georgetown,Marist,5,0.000000,1686.82,175,80.000000,...,35.187876,29.572099,71.468771,10.727273,15.727273,14.333333,23.363636,7.151515,2.303030,A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
956,2025,3163,3425,Connecticut,USC,2,80.000000,2340.59,175,91.176471,...,36.157871,28.984097,72.458461,12.612903,18.419355,7.580645,21.096774,7.516129,2.935484,A
957,2025,3395,3400,TCU,Texas,2,0.000000,2030.52,175,91.176471,...,38.388926,28.353663,70.154824,9.647059,19.441176,7.735294,18.382353,6.794118,2.647059,B
958,2025,3163,3417,Connecticut,UCLA,2,80.000000,2347.25,175,91.176471,...,34.902891,30.227762,71.788223,11.468750,14.468750,8.500000,18.093750,7.218750,2.875000,A
959,2025,3376,3400,South Carolina,Texas,1,100.000000,2423.61,175,90.909091,...,38.388926,28.353663,70.154824,9.647059,19.441176,7.735294,18.382353,6.794118,2.647059,A




W Mean Accuracy: 0.7673706298439774
W Mean Log Loss:  0.48009084708579836


100%|██████████| 68/68 [00:03<00:00, 21.73it/s]


,Season,TeamA,TeamB,TeamNameA,TeamNameB,ASeedNum,ALastTournPct,AELO,AMasseyRank,ARegSznPct,...,BOppFT,BOppAst,BOppTO,BOppOR,BOppDR,BOppStl,BOppBlk,AProb,BProb,Pred
0,2026,3104,3113,Alabama,Arizona St,6,50.0,1986.78,175,69.696970,...,73.977729,12.617647,18.470588,8.117647,22.705882,7.794118,2.705882,0.794318,0.205682,A
1,2026,3104,3124,Alabama,Baylor,6,50.0,1986.78,175,69.696970,...,65.169748,11.343750,14.656250,9.312500,21.031250,7.625000,2.468750,0.485530,0.514470,B
2,2026,3104,3155,Alabama,Clemson,6,50.0,1986.78,175,69.696970,...,71.512939,10.468750,13.500000,7.000000,21.281250,5.406250,3.218750,0.667855,0.332145,A
3,2026,3104,3158,Alabama,Col Charleston,6,50.0,1986.78,175,69.696970,...,67.961811,12.933333,16.966667,9.533333,23.000000,5.000000,2.800000,0.849187,0.150813,A
4,2026,3104,3160,Alabama,Colorado,6,50.0,1986.78,175,69.696970,...,70.043299,10.666667,16.909091,6.939394,20.090909,8.454545,3.484848,0.631521,0.368479,A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2273,2026,3452,3465,West Virginia,Cal Baptist,4,50.0,2084.59,175,81.818182,...,71.308776,12.424242,14.636364,8.515152,26.606061,6.636364,2.636364,0.856108,0.143892,A
2274,2026,3452,3471,West Virginia,UC San Diego,4,50.0,2084.59,175,81.818182,...,74.890767,12.548387,18.451613,9.548387,26.483871,6.709677,3.419355,0.870335,0.129665,A
2275,2026,3453,3465,WI Green Bay,Cal Baptist,13,0.0,1752.54,175,75.000000,...,71.308776,12.424242,14.636364,8.515152,26.606061,6.636364,2.636364,0.444540,0.555460,B
2276,2026,3453,3471,WI Green Bay,UC San Diego,13,0.0,1752.54,175,75.000000,...,74.890767,12.548387,18.451613,9.548387,26.483871,6.709677,3.419355,0.506018,0.493982,A


## Building the Submission DataFrame

The Kaggle submission requires a row for **every possible matchup** across all D1 teams — not just tournament teams. This cell handles the non-tournament pairs:

- `set_impossible_games_probs` identifies all team pairs where at least one team is not in the tournament and assigns them a neutral **0.5 probability** (these games can never occur, so the prediction doesn't affect the competition score)
- The tournament predictions from both pipeline runs are merged into `current_szn_df`
- All rows are combined into `final_df`, sorted by `ID`, and displayed

In [14]:
def set_game_id(row):
    return f"{row['Season']}_{row['TeamA']}_{row['TeamB']}"


def set_impossible_games_probs(gender, current_szn_df, teams_df):
    teams_a = current_szn_df["TeamA"].unique().tolist()
    teams_b = current_szn_df["TeamB"].unique().tolist()
    tourney_teams = set(teams_a + teams_b)

    teams = teams_df[teams_df["Season"] == season]["TeamID"].unique().tolist()

    data = []

    for idx, team_a in enumerate(teams):
        for idx_b in range(idx + 1, len(teams)):
            team_b = teams[idx_b]

            if team_a in tourney_teams and team_b in tourney_teams:
                continue

            teams_info_df = m_teams_info_df if gender == "M" else w_teams_info_df
            # There is no reason for setting team names or probabilities if these matchups will never happen in the tournament
            team_a_name = None  # set_team_name(team_a, teams_info_df)
            team_b_name = None  # set_team_name(team_b, teams_info_df)

            game_id = f"{season}_{team_a}_{team_b}"
            data.append([game_id, team_a, team_b, team_a_name, team_b_name, 0.5])

    data_df = pd.DataFrame(
        data, columns=["ID", "TeamA", "TeamB", "TeamNameA", "TeamNameB", "Pred"]
    )
    return data_df


current_szn_df = pd.concat([m_current_szn_df, w_current_szn_df], axis=0).reset_index(
    drop=True
)
current_szn_df["ID"] = current_szn_df.apply(lambda row: set_game_id(row), axis=1)

m_impossible_df = set_impossible_games_probs("M", m_current_szn_df, m_teams_df)
w_impossible_df = set_impossible_games_probs("W", w_current_szn_df, w_teams_df)

final_df = current_szn_df.loc[
    :, ["ID", "TeamA", "TeamB", "TeamNameA", "TeamNameB", "AProb"]
].rename({"AProb": "Pred"}, axis=1)
final_df = (
    pd.concat([final_df, m_impossible_df, w_impossible_df], axis=0)
    .sort_values(by="ID")
    .reset_index(drop=True)
)

display(final_df)

,ID,TeamA,TeamB,TeamNameA,TeamNameB,Pred
0,2026_1101_1102,1101,1102,None,None,0.5
1,2026_1101_1103,1101,1103,None,None,0.5
2,2026_1101_1104,1101,1104,None,None,0.5
3,2026_1101_1105,1101,1105,None,None,0.5
4,2026_1101_1106,1101,1106,None,None,0.5
...,...,...,...,...,...,...
132128,2026_3478_3480,3478,3480,None,None,0.5
132129,2026_3478_3481,3478,3481,None,None,0.5
132130,2026_3479_3480,3479,3480,None,None,0.5
132131,2026_3479_3481,3479,3481,None,None,0.5


## Matchup Lookup (Ad-hoc)

A convenience cell for inspecting the model's predicted win probability for any specific head-to-head matchup. Edit `team_a` and `team_b` to query any pairing from `final_df`.

In [15]:
team_a = "Baylor"
team_b = "Duke"

game = final_df[
    (
        (final_df["TeamNameA"].str.contains(team_a))
        & (final_df["TeamNameB"].str.contains(team_b))
    )
    | (
        (final_df["TeamNameB"].str.contains(team_a))
        & (final_df["TeamNameA"].str.contains(team_b))
    )
]

display(game)

,ID,TeamA,TeamB,TeamNameA,TeamNameB,Pred
73532,2026_3124_3181,3124,3181,Baylor,Duke,0.466831


In [16]:
def create_submission_file(df):
    df_filtered = df.loc[:, ["ID", "Pred"]]
    df_filtered.to_csv("submission.csv", index=False)


create_submission_file(final_df)